In [ ]:
#|hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai  # upgrade fastai on colab

In [ ]:
#|default_exp vision.rawpy

# RAW Image Processing

> Helpers for working with camera RAW files using rawpy

In [ ]:
#|export 
from __future__ import annotations
from fastai.basics import *
from fastai.vision.all import *
from fastai.data.transforms import *

import rawpy
import numpy as np
from PIL import Image
from functools import partial

try:
    import cv2
    cv2.setNumThreads(0)
except: pass

In [ ]:
#|hide
from nbdev.showdoc import *

In [ ]:
#|export
_all_ = ['RawpyImage', 'RAWImage', 'TensorRawImage', 'get_raw_files', 'default_rawpy_params', 
         'RAWImageBlock', 'show_raw_image']

## Helper Functions

In [ ]:
#|export
def get_raw_files(path, recurse=True, folders=None):
    "Get RAW files in `path` recursively, only in `folders`, if specified."
    raw_extensions = ['.cr2', '.cr3', '.nef', '.arw', '.dng', '.raf', '.orf', '.rw2', '.3fr']
    return get_files(path, extensions=raw_extensions, recurse=recurse, folders=folders)

In [ ]:
#|export
def default_rawpy_params():
    "Default RAW postprocessing parameters optimized for machine learning"
    return rawpy.Params(
        gamma=(1, 1),                    # Linear gamma for ML
        no_auto_bright=True,             # Consistent brightness
        output_color=rawpy.ColorSpace.sRGB,
        output_bps=16,                   # 16-bit output
        user_flip=0,                     # No rotation
        use_camera_wb=True,              # Camera white balance
        auto_bright_thr=0.0,             # No auto brightness
    )

In [ ]:
#|export
def show_raw_image(im, ax=None, figsize=None, title=None, ctx=None, cmap='viridis', 
                   norm=None, aspect=None, interpolation=None, alpha=None, vmin=None, vmax=None, **kwargs):
    "Show a RAW image on `ax`."
    # Convert RawpyImage to displayable format
    if hasattr(im, 'ndarr'):
        data = im.ndarr
        # Normalize for display if values > 1
        if data.max() > 1.0:
            data = data / data.max()
    else:
        data = im
    
    return show_image(data, ax=ax, figsize=figsize, title=title, ctx=ctx, cmap=cmap,
                     norm=norm, aspect=aspect, interpolation=interpolation, alpha=alpha, 
                     vmin=vmin, vmax=vmax, **kwargs)

## Core Classes

In [ ]:
#|export
class RawpyImage:
    "Core RAW image class - equivalent to PIL.Image.Image but for RAW files"
    def __init__(self, fn: Path, params: rawpy.Params):
        fn = str(fn)
        try:
            with rawpy.imread(fn) as raw_obj:
                self.obj = raw_obj
                self.ndarr = np.array(raw_obj.postprocess(params)).astype(np.float32)
                self._size = self.ndarr.shape[:2]
                self.mode = "RGB" if self.ndarr.shape[2] == 3 else f"{self.ndarr.shape[2]}ch"
        except rawpy._rawpy.LibRawIOError:
            print(f"Error: LibRawIOError: Could not open RAW file: {fn}")
            # Create a placeholder to prevent crashes
            self.ndarr = np.zeros((100, 100, 3), dtype=np.float32)
            self._size = (100, 100)
            self.mode = "RGB"

    def update_size(self):
        "Update size after image manipulation"
        self._size = self.ndarr.shape[:2]

    @property
    def width(self):
        return self.size[1]

    @property
    def height(self):
        return self.size[0]

    @property
    def size(self):
        return self._size

    @property
    def shape(self):
        return self.ndarr.shape

In [ ]:
#|export
class TensorRawImage(TensorImage):
    "Tensor subclass for RAW images with enhanced bit depth support"
    _show_args = {'cmap': 'viridis'}
    
    @classmethod
    def create(cls, fn: Path, params: rawpy.Params = None):
        "Create TensorRawImage from RAW file"
        if params is None: params = default_rawpy_params()
        raw_img = RawpyImage(fn, params)
        # Normalize to 0-1 range for tensor operations
        tensor_data = torch.from_numpy(raw_img.ndarr / 65535.0 if raw_img.ndarr.max() > 1.0 else raw_img.ndarr)
        # Ensure CHW format: (channels, height, width)
        if tensor_data.dim() == 3:
            tensor_data = tensor_data.permute(2, 0, 1)
        return cls(tensor_data)

In [ ]:
#|export
class RAWImage(RawpyImage):
    "Main RAW image class for fastai - equivalent to PILImage but for RAW files"
    _show_args = {}
    _tensor_cls = TensorRawImage
    
    @classmethod
    def create(cls, fn: Path, params: rawpy.Params = None):
        "Create RAWImage from file path with optional processing parameters"
        if params is None: params = default_rawpy_params()
        return cls(fn, params)
    
    def show(self, ctx=None, **kwargs): 
        "Show RAW image using fastai's display system"
        return show_raw_image(self, ctx=ctx, **kwargs)
        
    def __repr__(self): 
        return f'{self.__class__.__name__} mode={self.mode} size={"x".join([str(d) for d in self.size])}'

# Set up tensor class relationship
RAWImage._tensor_cls = TensorRawImage

## Transform Integration

In [ ]:
#|export
@ToTensor
def encodes(self, o:RAWImage): 
    "Convert RAWImage to TensorRawImage"
    tensor_data = torch.from_numpy(o.ndarr).float()
    # Normalize to 0-1 range if needed
    if tensor_data.max() > 1.0:
        tensor_data = tensor_data / 65535.0
    # Ensure CHW format
    if tensor_data.dim() == 3:
        tensor_data = tensor_data.permute(2, 0, 1)
    return TensorRawImage(tensor_data)

In [ ]:
#|export
def RAWImageBlock(params=None):
    "A `TransformBlock` for RAW images"
    if params is None: params = default_rawpy_params()
    return TransformBlock(
        type_tfms=partial(RAWImage.create, params=params), 
        batch_tfms=IntToFloatTensor
    )

## RAW-Specific Processing Functions

In [ ]:
#|export
@patch
def exposure_adjust(self:TensorRawImage, stops:float):
    "Adjust exposure by number of stops (+1 = double brightness, -1 = half brightness)"
    return TensorRawImage(self * (2 ** stops))

In [ ]:
#|export
@patch
def highlight_recovery(self:TensorRawImage, threshold:float=0.95, strength:float=0.5):
    "Recover blown highlights in RAW image"
    # Create mask for overexposed areas
    mask = self > threshold
    # Blend overexposed areas with threshold
    recovered = torch.where(mask, 
                           threshold + (self - threshold) * strength, 
                           self)
    return TensorRawImage(recovered)

In [ ]:
#|export
@patch
def shadow_lift(self:TensorRawImage, threshold:float=0.1, strength:float=0.3):
    "Lift shadows in RAW image"
    # Create mask for shadow areas
    mask = self < threshold
    # Lift shadows
    lifted = torch.where(mask,
                        self + (threshold - self) * strength,
                        self)
    return TensorRawImage(lifted)

## Testing and Examples

In [ ]:
#|export
def test_raw_creation():
    "Test basic RAW image creation (requires test file)"
    # This would need a real RAW file to test
    # For now, just test parameter creation
    params = default_rawpy_params()
    assert isinstance(params, rawpy.Params)
    assert params.gamma == (1, 1)
    assert params.no_auto_bright == True
    print("✓ RAW parameter creation test passed")

In [ ]:
#|export
def test_raw_files_discovery():
    "Test RAW file discovery function"
    # Test with a dummy path
    files = get_raw_files("/tmp", recurse=False)  # Should return empty list for non-existent path
    assert isinstance(files, list)
    print("✓ RAW file discovery test passed")

In [ ]:
# Run basic tests
test_raw_creation()
test_raw_files_discovery()

## Usage Examples

Here's how to use the RAW image processing:

```python
from fastai.vision.all import *
from fastai.vision.rawpy import *

# Load RAW files from a directory
raw_files = get_raw_files("path/to/raw/images")

# Create a RAW image with custom processing
params = default_rawpy_params()
params.gamma = (2.2, 4.5)  # Standard gamma curve
raw_img = RAWImage.create(raw_files[0], params)

# Show the image
raw_img.show()

# Create a DataBlock for RAW images
photos = DataBlock(
    blocks=(RAWImageBlock(), CategoryBlock),
    get_x=ColReader('filename'),
    get_y=ColReader('label'),
    splitter=RandomSplitter(),
    batch_tfms=aug_transforms(size=224)
)

# Create dataloaders
dls = photos.dataloaders(df)
dls.show_batch()
```

## Export -

In [ ]:
#|hide
from nbdev import nbdev_export
nbdev_export()